# Features

Cloud cover and cloud height from EUMetSat data archive spanning 2024 year.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import math as maths
import torch, torch.nn as nn
import pandas as pd
import copy

torch.cuda.is_available()

True

In [2]:
sat_data = np.load("data/ireland_clouds_20240101_20250101.npz")
print(sat_data.files)

['times', 'cloud_mask', 'cth_km', 'lat', 'lon']


In [3]:
csv_name = ["hly532.csv","hly3904_subset.csv", "hly4935_subset.csv", "hly518_subset.csv", "hly3723_subset.csv"]

dates = []

def get_target_data(csv_name):
    data = pd.read_csv(csv_name,skiprows=41, low_memory=False)
    data = data.rename(columns={'ind': 'Rainfall_indicator', "ind.1": "Dry_bulb_indicator", "ind.2": "Wet_bulb_indicator", "ind.3": "wind_speed_indicator", "ind.4": "wind_dir_indicator"})
    numeric_like_cols = ["vappr", "rhum", "wddir", "vis", "clht", "clamt"]
    data[numeric_like_cols] = data[numeric_like_cols].apply(pd.to_numeric, errors="coerce")
    data[numeric_like_cols] = data[numeric_like_cols].ffill().bfill()

    data["date"] = pd.to_datetime(data["date"], format="%d-%b-%Y %H:%M")
    mask = data["date"].dt.year == 2024
    data = data[mask]
    target = data["clamt"].values
    print(f"{csv_name} target shape: {target.shape}")
    dates.append(data["date"].values)

    return target




target_data = []

for csv in csv_name:
    target_data.append(get_target_data(csv))

dates = np.array(dates).T
print(f"dates shape: {dates.shape}")

for date in dates:
    assert np.all(date[0] == date[1 :]), "Dates are not the same across all CSV files."



hly532.csv target shape: (8784,)
hly3904_subset.csv target shape: (8784,)
hly4935_subset.csv target shape: (8784,)
hly518_subset.csv target shape: (8784,)
hly3723_subset.csv target shape: (8784,)
dates shape: (8784, 5)


In [4]:
target_data = np.array(target_data)
print(f"target_data shape: {target_data.shape}")


target_data shape: (5, 8784)


In [5]:
cloud_mask = sat_data["cloud_mask"]
cth_km = sat_data["cth_km"]
sat_times = sat_data["times"]

# station rows must be the same hours in the same order as the satellite frames (all UTC)
assert np.array_equal(dates[:, 0].astype("datetime64[us]"), sat_times)

# one True/False per HOUR: missing only if the WHOLE frame is "no data" (3)
sat_ok = ~(cloud_mask == 3).all(axis=(1, 2))                 # (8784,)

# target_data is (stations, hours): an hour is usable only if EVERY station has a cloud amount,
# so collapse the station axis -> one value per hour
target_ok = ~np.isnan(target_data).any(axis=0)               # (8784,)

hour_ok = sat_ok & target_ok
print(f"hours: {len(hour_ok)}, missing: {(~hour_ok).sum()}")
print("missing hours:", sat_times[~hour_ok])

hours: 8784, missing: 14
missing hours: ['2024-06-27T07:00:00.000000' '2024-07-23T22:00:00.000000'
 '2024-07-23T23:00:00.000000' '2024-07-24T00:00:00.000000'
 '2024-07-24T01:00:00.000000' '2024-07-24T02:00:00.000000'
 '2024-07-24T03:00:00.000000' '2024-07-24T04:00:00.000000'
 '2024-08-17T13:00:00.000000' '2024-08-17T14:00:00.000000'
 '2024-08-17T15:00:00.000000' '2024-08-17T16:00:00.000000'
 '2024-08-17T17:00:00.000000' '2024-09-04T08:00:00.000000']


In [6]:
# Build 2 channels per timestep -> frames: (hours, 2, 90, 120)
#   channel 0: cloud      1 = cloud, 0 = clear (clear sea / clear land / no-data pixel)
#   channel 1: height_km  cloud top height in km, 0 where clear
# Sea vs land never changes, so it's not worth a channel of its own.

cloud = (cloud_mask == 2).astype(np.float16)

# cth_km is NaN only in the 14 missing hours (those samples get dropped anyway) -> 0.
# CTH comes on a coarser grid, so it spills onto some clear pixels -- zero it wherever
# the cloud mask says clear so the two channels agree.
height_km = np.nan_to_num(cth_km, nan=0.0) * cloud

frames = np.stack([cloud, height_km], axis=1)    # float16 to save memory (~380 MB)



print(f"frames shape: {frames.shape}  (hours, channels, lat, lon)")
print(f"height range: {height_km.min():.2f} - {height_km.max():.2f} km")

import gc
del cloud, height_km, cth_km, sat_ok
gc.collect()

frames shape: (8784, 2, 90, 120)  (hours, channels, lat, lon)
height range: 0.00 - 15.04 km


65

In [7]:
K = 6   # hours of satellite history per sample
H = 3   # predict cloud amount this many hours after the last input frame

# Don't delete the missing rows from the arrays -- that would join hours either side of a
# gap into one "continuous" sequence. Keep the full hourly arrays and instead list the
# samples that are valid: a sample ending at hour t uses frames t-K+1..t and target t+H,
# and is kept only if all of those hours are present.
sample_t = np.array([t for t in range(K - 1, len(hour_ok) - H)
                     if hour_ok[t - K + 1:t + 1].all() and hour_ok[t + H]])

print(f"possible samples: {len(hour_ok) - H - K + 1}, kept: {len(sample_t)}, "
      f"dropped: {len(hour_ok) - H - K + 1 - len(sample_t)}")

# e.g. inputs and target for the first valid sample
t = sample_t[0]
X0 = frames[t - K + 1:t + 1]        # (K, 2, 90, 120): K timesteps x 2 channels
y0 = target_data[:, t + H]          
print(X0.shape, y0)

clamt_now    = target_data[:, sample_t].T        # (samples, stations) cloud amount at the last input hour t
clamt_future = target_data[:, sample_t + H].T    # (samples, stations) cloud amount at t + H
y = clamt_future - clamt_now                     # (samples, stations) change in oktas over the next H hours
print(f"y shape: {y.shape}  (samples, stations)")


possible samples: 8776, kept: 8734, dropped: 42
(6, 2, 90, 120) [1. 7. 3. 7. 1.]
y shape: (8734, 5)  (samples, stations)


In [8]:
week_hours = 24 * 7
start_week = (sample_t - K + 1) // week_hours     # week of the sample's first input frame
end_week   = (sample_t + H) // week_hours         # week of its target hour
inside = start_week == end_week                   # the whole sample (inputs + target) sits in one week

block = start_week % 7                            # every 7th week -> test, the next -> val, the rest -> train
idx = np.arange(len(sample_t))
train_idx = idx[inside & (block >= 2)]
val_idx   = idx[inside & (block == 1)]
test_idx  = idx[inside & (block == 0)]
print(len(train_idx), len(val_idx), len(test_idx), "dropped at week edges:", (~inside).sum())

5780 1265 1273 dropped at week edges: 416


In [9]:
def channel_stats(frames, hours, chunk=256):
    C = frames.shape[1]
    s, ss, count = np.zeros(C), np.zeros(C), 0
    for j in range(0, len(hours), chunk):
        b = frames[hours[j:j + chunk]].astype(np.float64)   # (b, C, 90, 120)
        s  += b.sum(axis=(0, 2, 3))                         # sum over everything except channel
        ss += (b ** 2).sum(axis=(0, 2, 3))
        count += b[:, 0].size
    mean = s / count
    std = np.sqrt(ss / count - mean ** 2)
    return mean, np.where(std == 0, 1.0, std)

# every input hour touched by a training sample (first frame of the first sample -> last frame of the last)
train_hours = np.arange(sample_t[train_idx[0]] - K + 1, sample_t[train_idx[-1]] + 1)
train_hours = train_hours[hour_ok[train_hours]]             # skip the missing hours

x_mean, x_std = channel_stats(frames, train_hours)
y_mean, y_std = y[train_idx].mean(axis=0), y[train_idx].std(axis=0)


In [10]:
class FrameDataset(torch.utils.data.Dataset):
    def __init__(self, frames, y, sample_t, idx, K):
        self.frames, self.y, self.sample_t, self.idx, self.K = frames, y, sample_t, idx, K
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        t = self.sample_t[j]
        x = self.frames[t - self.K + 1:t + 1]
        return torch.from_numpy(x.astype(np.float32)), torch.from_numpy(self.y[j].astype(np.float32))

train_loader = torch.utils.data.DataLoader(FrameDataset(frames, y, sample_t, train_idx, K), batch_size=32, shuffle=True)
val_loader   = torch.utils.data.DataLoader(FrameDataset(frames, y, sample_t, val_idx, K),   batch_size=64)
test_loader  = torch.utils.data.DataLoader(FrameDataset(frames, y, sample_t, test_idx, K),  batch_size=64)


In [11]:
###
#Failed attemp to use ConvLSTM from github
###
"""from class_convlstm import ConvLSTM

convlstm_layer = []
img_size_list = [(90, 120)]
num_layers = 3
input_channel = 2
hidden_channel = 64
kernel_size = (3, 3)
stride = (1, 1)
padding = (1, 1)
for i in range(num_layers):
    convlstm_layer.append(ConvLSTM(input_channel=input_channel, hidden_channel=hidden_channel, kernel_size=kernel_size,
                  stride=stride, padding=padding, batch_first=True)
    )
    input_channel = hidden_channel
"""

input_channel = 2
kernel_size = (3, 3)
stride = (1, 1)
padding = (1, 1)



class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)    # matches out_channels
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.gelu(self.bn(self.conv(x)))   # uses the names defined above




In [ ]:
class CloudCNN(nn.Module):
    def __init__(self, K, n_channels, n_stations):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(K * n_channels, 16, stride=2),   # (B, 12, 90, 120) -> (B, 16, 45, 60)
            ConvBlock(16, 32, stride=2),               # -> (B, 32, 23, 30)
            ConvBlock(32, 64, stride=2),               # -> (B, 64, 12, 15)
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((6, 8)),              # -> (B, 64, 6, 8)  keeps WHERE the cloud is
            nn.Flatten(),                              # -> (B, 3072)
            nn.Dropout(0.3),
            nn.Linear(64 * 6 * 8, n_stations),         # -> (B, 5)
        )

    def forward(self, x):
        B, K, C, H, W = x.shape
        x = x.reshape(B, K * C, H, W)                  # stack the K hours into the channel axis
        return self.head(self.features(x))


In [13]:
class ScaledModel(nn.Module):
    def __init__(self, net, x_mean, x_std, y_mean, y_std):
        super().__init__()
        self.net = net
        f = lambda a: torch.tensor(a, dtype=torch.float32)
        self.register_buffer("x_mean", f(x_mean).view(1, 1, -1, 1, 1))   # broadcasts over (B, K, C, H, W)
        self.register_buffer("x_std",  f(x_std).view(1, 1, -1, 1, 1))
        self.register_buffer("y_mean", f(y_mean).view(1, -1))            # broadcasts over (B, stations)
        self.register_buffer("y_std",  f(y_std).view(1, -1))

    def scale_y(self, y):                 # training targets -> scaled
        return (y - self.y_mean) / self.y_std

    def forward_scaled(self, x_raw):      # raw frames -> scaled change (training)
        return self.net((x_raw - self.x_mean) / self.x_std)

    def forward(self, x_raw):             # raw frames -> change in oktas (test)
        return self.forward_scaled(x_raw) * self.y_std + self.y_mean


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ScaledModel(CloudCNN(K, frames.shape[1], y.shape[1]), x_mean, x_std, y_mean, y_std).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")


parameters: 40,469


In [14]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
num_epochs, patience = 50, 8
best_val, best_state, bad_epochs = float("inf"), None, 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model.forward_scaled(xb), model.scale_y(yb))
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_loss += criterion(model.forward_scaled(xb), model.scale_y(yb)).item() * len(xb)
    val_loss /= len(val_loader.dataset)
    scheduler.step(val_loss)

    if val_loss < best_val:
        best_val, best_state, bad_epochs = val_loss, copy.deepcopy(model.state_dict()), 0
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"early stop at epoch {epoch + 1}")
            break
    print(f"epoch {epoch + 1}: train {train_loss:.4f}  val {val_loss:.4f}  (scaled MSE)")

model.load_state_dict(best_state)
print(f"best val: {best_val:.4f}")


epoch 1: train 1.0041  val 0.9744  (scaled MSE)
epoch 2: train 0.9258  val 0.9532  (scaled MSE)
epoch 3: train 0.8817  val 0.9458  (scaled MSE)
epoch 4: train 0.8502  val 0.9322  (scaled MSE)
epoch 5: train 0.8069  val 0.9424  (scaled MSE)
epoch 6: train 0.7601  val 0.9751  (scaled MSE)
epoch 7: train 0.7177  val 0.9612  (scaled MSE)
epoch 8: train 0.6720  val 1.0085  (scaled MSE)
epoch 9: train 0.5879  val 0.9911  (scaled MSE)
epoch 10: train 0.5505  val 1.0045  (scaled MSE)
epoch 11: train 0.5266  val 1.0080  (scaled MSE)
early stop at epoch 12
best val: 0.9322


In [15]:
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds.append(model(xb.to(device)).cpu().numpy())    # plain forward -> change in oktas
# forecast = cloud amount now + predicted change, kept inside 0-8
preds = np.clip(clamt_now[test_idx] + np.concatenate(preds), 0, 8)
y_true = clamt_future[test_idx]

# baseline 1: persistence = the cloud amount at t stays the same until t + H
persistence = clamt_now[test_idx]

# baseline 2: climatology = each station's average cloud amount, TRAINING samples only
climatology = np.broadcast_to(clamt_future[train_idx].mean(axis=0), y_true.shape)

# baseline 3: average cloud amount at that hour of day, TRAINING samples only
hour_of_day = sat_times[sample_t + H].astype("datetime64[h]").astype(int) % 24
hourly_mean = np.array([clamt_future[train_idx][hour_of_day[train_idx] == h].mean(axis=0) for h in range(24)])
diurnal = hourly_mean[hour_of_day[test_idx]]

rmse = lambda a, b: np.sqrt(((a - b) ** 2).mean(axis=0))
exact = lambda a, b: (np.rint(a) == b).mean()               # fraction of forecasts right to the okta
for name, p in [("model", preds), ("persistence", persistence), ("climatology", climatology), ("hour-of-day mean", diurnal)]:
    print(f"{name:18s} RMSE per station {rmse(p, y_true).round(3)}  overall {np.sqrt(((p - y_true) ** 2).mean()):.3f}"
          f"  exact okta {exact(p, y_true):.2f}")


model              RMSE per station [1.564 1.628 1.706 1.722 1.881]  overall 1.704  exact okta 0.41
persistence        RMSE per station [1.668 1.737 1.786 1.787 2.001]  overall 1.799  exact okta 0.51
climatology        RMSE per station [2.093 2.32  1.837 2.288 2.255]  overall 2.166  exact okta 0.12
hour-of-day mean   RMSE per station [2.066 2.303 1.645 2.273 2.237]  overall 2.119  exact okta 0.19
